# Pablo Prol Prieto and Pablo Sala Fernández (Team Pablo(s) in Kaggle)
## Session 3 notebook

### Imports

In [96]:
import numpy as np 
import pandas as pd 

import torch 
import seaborn as sns
import matplotlib.pyplot as plt

### Read data

In [97]:
train_set = pd.read_csv('train_project3.csv')
test_set = pd.read_csv('test_project3.csv')

### Explore data

In [ ]:
train_set.describe()

In [ ]:
train_set.info()

In [ ]:
# Drop outliers ('Annual_Premium' > 100000)
train_set = train_set[train_set['Annual_Premium'] < 100000]

In [ ]:
train_set.head(10)

We will consider clients who are interested in getting a car insurance (Rows with 'Response' = 1) to be the positive cases and the rest to be the negative cases.

We will now compare the number of positive and negative cases to see how much imbalance there is in our train set.

In [ ]:
def compute_case_frequencies(columns):
    
    labels = np.array(columns)
    
    N = labels.shape[0]
    
    positive_frequencies = np.sum(columns,axis = 0) / N
    negative_frequencies = 1 - positive_frequencies
    
    return positive_frequencies, negative_frequencies

In [ ]:
freq_pos, freq_neg = compute_case_frequencies(train_set['Response'])
df = pd.DataFrame({"Cases": ['0'], "Label": ["Negative"], "Value": [freq_neg]})
df2 = pd.DataFrame({"Cases": ['1'], "Label": ["Positive"], "Value": [freq_pos]})
df = pd.concat([df, df2], ignore_index=True)
sns.barplot(x="Cases", y="Value" ,data=df)

In [ ]:
positive_cases = train_set['Response'].value_counts()[1]
negative_cases = train_set['Response'].value_counts()[0]
total_cases = len(train_set)

positive_percentage = (positive_cases / total_cases) * 100
negative_percentage = (negative_cases / total_cases) * 100

print("Positive cases percentage: {:.2f}%".format(positive_percentage))
print("Negative cases percentage: {:.2f}%".format(negative_percentage))

The binary cross-entropy (BCE) is defined as:

$$
\sum_{i=1}^{N}l(y_i,p(x_i))=\sum_{i=1}^{N}-(y_i \cdot \log(p(x_i))+(1-y_i) \cdot \log(1-p(x_i)))
$$

where $y_i$ is the target variable (can only be 0 or 1), $x_i$ are the variables/features, and $p$ the probability model of being of class 1.

You can see that with a sample of the class 1:

$$
l(y_i=1,p(x_i))=-\log(p(x_i))
$$

and thus the loss is 0 when $p(x_i)=1$ (the sample is of class 1, and the model outputs a probability 1 of being of class 1), and the loss is $\infty$ when $p(x_i)=0$ (the sample is of class 1, and the model outputs a probability 0 of being of class 1).

On the other hand, when the sample is of the class 0:

$$
l(y_i=0,p(x_i))=-\log(1-p(x_i))
$$

and thus the loss is 0 when $p(x_i)=0$ (the sample is of class 0, and the model outputs a probability 0 of being of class 1), and the loss is $\infty$ when $p(x_i)=1$ (the sample is of class 0, and the model outputs a probability 1 of being of class 1).

Let's plot the value of the number of positive and negative cases again to see whether it is indeed balanced.

use the Pipeline and ColumnTransformer classes from sklearn to create a preprocessing pipeline that transforms certain features to categorical and then applies one-hot encoding to all categorical features.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Define the columns to be transformed to categorical
categorical_features = ['Driving_License', 'Region_Code', 'Previously_Insured', 'Policy_Sales_Channel', 'Gender', 'Vehicle_Age', 'Vehicle_Damage']

# Create a pipeline that transforms the features to categorical and then applies one-hot encoding
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ], remainder='passthrough'  # leave the rest of the columns untouched
)

Building a Boosting Classifier from scratch involves several steps. Here's a step-by-step plan:

- Initialize weights for each sample in the dataset to be equal.
- Train a weak classifier (DecisionTreeClassifier) on the dataset.
- Calculate the error of the weak classifier.
- Calculate the classifier's weight in the ensemble based on its error.
- Update the weights of the samples in the dataset. Misclassified samples get their weights increased, correctly classified samples get their weights decreased.
- Normalize the weights.
- Repeat steps 2-6 for a specified number of iterations (or until error is low enough).
- To make a prediction, run each classifier on the input and combine their predictions (weighted by their individual weights in the ensemble).

Here's the Python code for a simple Boosting Classifier:

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import class_weight
import numpy as np

class WeightedBoostingClassifier:
    def __init__(self, n_estimators=50, learning_rate=0.05, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features=None):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.models = []
        self.model_weights = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        sample_weights = np.full(n_samples, 1/n_samples)

        # Compute class weights
        class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
        class_weights = dict(enumerate(class_weights))

        for _ in range(self.n_estimators):
            model = DecisionTreeClassifier(max_depth=self.max_depth, min_samples_split=self.min_samples_split, min_samples_leaf=self.min_samples_leaf, max_features=self.max_features, class_weight=class_weights)
            model.fit(X, y, sample_weight=sample_weights)
            predictions = model.predict(X)

            misclassified = predictions != y
            error = np.mean(np.average(misclassified, weights=sample_weights))
            if error == 0:
                break
            
            model_weight = self.learning_rate * 0.5 * np.log((1.0 - error) / error)

            sample_weights *= np.exp(model_weight * misclassified * ((sample_weights > 0) | (model_weight < 0)))
            sample_weights /= np.sum(sample_weights)

            self.models.append(model)
            self.model_weights.append(model_weight)

    def predict(self, X):
        model_predictions = np.array([model.predict(X) for model in self.models])
        return np.where(np.dot(self.model_weights, model_predictions) > 0, 1, 0)

This is a simple implementation and may not perform well on complex datasets. For more advanced features like handling missing values, different loss functions, etc., consider using a mature library like scikit-learn or XGBoost.

In [ ]:
from xgboost import XGBClassifier


# Define your features and target variable
y = train_set['Response']
X = train_set.drop('Response', axis=1)

# Fit the preprocessor and transform the training data
X_transformed = preprocessor.fit_transform(X)

# Initialize the model
model = WeightedBoostingClassifier(n_estimators=50)

# Fit the model
model.fit(X_transformed, y)

# Transform the test data
X_test = test_set
X_test_transformed = preprocessor.transform(X_test)

# Predict the test set
y_test = model.predict(X_test_transformed)

# Save our predictions
test_set['Response'] = y_test
test_set[['id','Response']].to_csv('submission.csv', index=None)